# AIMO3 Optimized GPU Inference - Production Ready

## Key Optimizations:
- ✅ Fixed concurrent.futures import issue
- ✅ H100-optimized vLLM parameters
- ✅ Enhanced batch processing with dynamic sizing
- ✅ Production monitoring & observability
- ✅ Memory-efficient KV cache management
- ✅ Optimized pooling kernel for faster inference
- ✅ Improved error handling & recovery

Based on TIR (Tool-Integrated Reasoning) with H100 GPU optimizations

In [1]:
import time
import numpy as np
import os
from typing import Dict, List

# Competition time constraints
start_time = time.time()
final_cutoff_time = start_time + (4 * 60 + 55) * 60  # 4h 55m - safe margin

print(f"⏱️  Start time: {time.strftime('%H:%M:%S', time.localtime(start_time))}")
print(f"⏱️  Cutoff time: {time.strftime('%H:%M:%S', time.localtime(final_cutoff_time))}")

⏱️  Start time: 14:14:43
⏱️  Cutoff time: 19:09:43


## Environment Setup & Cleanup

In [2]:
import subprocess
import sys

# Async cleanup of unused packages to save memory
print("🧹 Cleaning up unused packages...")
uninstall_proc = subprocess.Popen(
    [sys.executable, "-m", "pip", "uninstall", "--yes", 
     "tensorflow", "matplotlib", "keras", "scikit-learn"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

🧹 Cleaning up unused packages...


In [ ]:
%%time
# Warm up filesystem cache for faster loading
print("🔥 Warming up filesystem cache...")
!find /kaggle/usr/lib -type f -print0 2>/dev/null | xargs -0 -P 32 -n 500 cat > /dev/null

🔥 Warming up filesystem cache...


## Optimized Model Cache Loading

In [ ]:
def cache_model(path, exts=(".bin", ".pt", ".safetensors"), num_workers=None, chunk_mb=1024):
    """Pre-read model weight files into OS page cache with H100 optimizations.
    
    Args:
        path: Model directory path
        exts: File extensions to cache
        num_workers: Parallel workers (defaults to CPU count)
        chunk_mb: Chunk size for reading (larger for H100)
    """
    import os
    import multiprocessing
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def warmup_file(fpath):
        """Read file into cache."""
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, "rb") as f:
                while True:
                    data = f.read(chunk_size)
                    if not data:
                        break
                    total += len(data)
        except Exception as e:
            print(f"⚠️  Error caching {os.path.basename(fpath)}: {e}")
            return fpath, 0
        return fpath, total

    # Collect files
    if os.path.isdir(path):
        files = [
            os.path.join(root, name)
            for root, _, names in os.walk(path)
            for name in names
            if name.endswith(exts)
        ]
        files.sort(key=lambda x: os.path.getsize(x), reverse=True)  # Large files first
    else:
        files = [path]

    if not files:
        raise ValueError(f"No model files found under: {path}")

    # Optimize worker count for H100
    if num_workers is None:
        try:
            num_workers = min(multiprocessing.cpu_count(), 16)
        except Exception:
            num_workers = 8

    print(f"📦 Caching {len(files)} file(s) with {num_workers} workers")
    t0 = time.time()
    total_bytes = 0

    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
            if i % 5 == 0 or i == len(files):
                print(f"  [{i}/{len(files)}] ✓ {total_bytes / 1024**3:.1f} GB cached")

    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    throughput = gb / elapsed if elapsed > 0 else 0
    print(f"✅ Cached {gb:.2f} GB in {elapsed:.1f}s ({throughput:.2f} GB/s)")
    return total_bytes


# Cache model weights - optimized for H100
cache_model("/kaggle/input/gpt-oss-120b/transformers/default/1", 
            num_workers=16, chunk_mb=1024)

In [ ]:
%%time
# Copy vLLM compile cache if available
import os
if os.path.exists("/kaggle/input/gpt-oss-120b-cache-compile/torch_compile_cache"):
    print("📋 Copying vLLM compile cache...")
    !mkdir -p /root/.cache/vllm/
    !cp -r /kaggle/input/gpt-oss-120b-cache-compile/torch_compile_cache /root/.cache/vllm/
    print("✅ Compile cache ready")
else:
    print("ℹ️  No pre-compiled cache found - will compile on first run")

In [ ]:
# Wait for cleanup to finish
print("⏳ Waiting for package cleanup...")
uninstall_proc.wait()
print("✅ Environment ready")

## Environment Variables - H100 Optimized

In [ ]:
# H100-optimized environment variables
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# H100-specific optimizations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Better memory management
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # Async kernel launches

print("✅ Environment configured for H100")

## Python Tool Integration

In [ ]:
%%writefile local_python_tool.py
"""Production-ready Python tool with error handling and monitoring."""
import os
import queue
import threading
import zmq
import cloudpickle
import time
from typing import Any, Optional


class PythonTool:
    """Thread-safe Python execution tool with monitoring."""
    
    def __init__(self, timeout: float = 20.0):
        self.timeout = timeout
        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REQ)
        
        # Generate unique address
        port_offset = threading.get_ident() % 1000
        self.address = f"tcp://localhost:{5555 + port_offset}"
        self.socket.connect(self.address)
        
        # Metrics
        self.exec_count = 0
        self.error_count = 0
        self.total_time = 0.0
        
    def execute(self, code: str) -> tuple[Any, Optional[str]]:
        """Execute Python code with timeout and error handling.
        
        Returns:
            (result, error_message)
        """
        self.exec_count += 1
        start = time.time()
        
        try:
            self.socket.send(cloudpickle.dumps({"code": code}))
            
            # Set socket timeout
            self.socket.setsockopt(zmq.RCVTIMEO, int(self.timeout * 1000))
            
            response = self.socket.recv()
            result = cloudpickle.loads(response)
            
            elapsed = time.time() - start
            self.total_time += elapsed
            
            if result.get("status") == "error":
                self.error_count += 1
                return None, result.get("error", "Unknown error")
            
            return result.get("result"), None
            
        except zmq.error.Again:
            self.error_count += 1
            return None, f"Timeout after {self.timeout}s"
        except Exception as e:
            self.error_count += 1
            return None, f"Execution error: {str(e)}"
        finally:
            elapsed = time.time() - start
            self.total_time += elapsed
    
    def get_stats(self) -> dict:
        """Get execution statistics."""
        return {
            "executions": self.exec_count,
            "errors": self.error_count,
            "total_time": self.total_time,
            "avg_time": self.total_time / max(self.exec_count, 1),
            "error_rate": self.error_count / max(self.exec_count, 1)
        }
    
    def cleanup(self):
        """Cleanup resources."""
        try:
            self.socket.close()
            self.context.term()
        except:
            pass

## Configuration & Constants

In [ ]:
import warnings
warnings.simplefilter('ignore')

import re
import math
import subprocess
import requests
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple
import json

# Import concurrent.futures at module level - FIX FOR THE ORIGINAL BUG
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, wait, FIRST_COMPLETED, as_completed

# Competition parameters
@dataclass
class Config:
    # Model configuration
    model_path: str = "/kaggle/input/gpt-oss-120b/transformers/default/1"
    tokenizer_path: str = "/kaggle/input/gpt-oss-120b/transformers/default/1"
    
    # vLLM H100-optimized parameters
    gpu_memory_utilization: float = 0.95  # H100 has 80GB, use most of it
    max_model_len: int = 12000  # Increased for H100
    tensor_parallel_size: int = 1
    dtype: str = "bfloat16"  # H100 optimized
    enable_chunked_prefill: bool = True  # Better memory efficiency
    max_num_batched_tokens: int = 8192  # H100 optimized
    max_num_seqs: int = 256  # Increased batch size
    
    # Inference parameters
    temperature: float = 0.7
    top_p: float = 0.95
    max_tokens: int = 8000
    n_samples: int = 8  # Parallel samples per question
    
    # TIR parameters
    max_rounds: int = 8
    python_timeout: float = 20.0
    
    # Resource management
    use_dynamic_budget: bool = True
    base_budget_per_question: int = 360  # seconds
    

config = Config()

print("⚙️  Configuration:")
print(f"  GPU Memory: {config.gpu_memory_utilization * 100}%")
print(f"  Max Length: {config.max_model_len}")
print(f"  Batch Size: {config.max_num_seqs}")
print(f"  Samples: {config.n_samples}")

## vLLM Server - H100 Optimized

In [ ]:
def start_vllm_server(config: Config) -> subprocess.Popen:
    """Start vLLM server with H100 optimizations."""
    import sys
    
    command = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", config.model_path,
        "--tokenizer", config.tokenizer_path,
        "--gpu-memory-utilization", str(config.gpu_memory_utilization),
        "--max-model-len", str(config.max_model_len),
        "--tensor-parallel-size", str(config.tensor_parallel_size),
        "--dtype", config.dtype,
        "--port", "8000",
        "--host", "127.0.0.1",
        
        # H100 optimizations
        "--max-num-batched-tokens", str(config.max_num_batched_tokens),
        "--max-num-seqs", str(config.max_num_seqs),
        "--enable-chunked-prefill",
        "--disable-log-requests",
        
        # Performance
        "--swap-space", "4",  # Minimal swap for H100
        "--max-parallel-loading-workers", "8",
    ]
    
    print("🚀 Starting vLLM server with H100 optimizations...")
    proc = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    return proc


# Start server
vllm_process = start_vllm_server(config)

## TIR Prompt Templates

In [ ]:
# Proven TIR prompt from baseline
TIR_PROMPT = """Please reason step by step and use the python tool to solve the math problem.
Finally, return only the verified final answer in \\boxed{}, where the answer is an integer in [0, 99999].
Never guess. Use the Python tool to verify all calculations.

Key guidelines:
1. Break down the problem into clear steps
2. Use Python for all numerical computations
3. Verify intermediate results
4. Double-check the final answer
5. Ensure answer is an integer in [0, 99999]
"""

print("✅ TIR prompt loaded")

## Python Tool Pool

In [ ]:
import queue
from local_python_tool import PythonTool

# Create thread pool of Python tools
K = config.n_samples
python_pool = queue.Queue(maxsize=K)

print(f"🔧 Initializing {K} Python execution environments...")
for i in range(K):
    tool = PythonTool(timeout=config.python_timeout)
    python_pool.put(tool)

print(f"✅ Python tool pool ready ({K} workers)")

In [ ]:
# Memory cleanup code for Python tools
import gc

CLEANUP_CODE = r"""
import gc
_keep = {
    k: v for k, v in globals().items()
    if k.startswith('_') or k in {'gc', 'math', 'numpy', 'np', 're', 'sympy'}
}
_clear = {k for k in globals() if k not in _keep}
for k in _clear:
    del globals()[k]
gc.collect()
"""

print("✅ Cleanup code ready")

## Production-Ready Inferencer with Monitoring